In [18]:
import sys
from pathlib import Path
import numpy as np
import matplotlib
SHOW = "--no-show" not in sys.argv
if not SHOW:
    matplotlib.use("Agg")  # sem janela; apenas salva os arquivos
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [19]:

# Pasta onde as imagens são salvas: a mesma do script (AULA_10/)
OUT_DIR = Path.cwd()

# ----------------------------------------------------------------------------
# Universos
# ----------------------------------------------------------------------------
x_imp = np.arange(0, 100.01, 0.5)
x_exp = np.arange(0, 10.001, 0.05)
x_urg = np.arange(0, 100.01, 0.1)


def construir_sistema(crit_mf="tri", defuzz="centroid"):
    """Monta o sistema Mamdani (9 regras).

    crit_mf : 'tri'  -> Urgência Crítica triangular (baseline)
              'trap' -> Urgência Crítica trapezoidal (Abordagem A)
    defuzz  : 'centroid' | 'mom' | ...
    """
    impacto = ctrl.Antecedent(x_imp, "impacto")
    explor = ctrl.Antecedent(x_exp, "explorabilidade")
    urg = ctrl.Consequent(x_urg, "urgencia", defuzzify_method=defuzz)

    # Impacto: Baixo (0-40), Médio (20-80), Alto (60-100)
    impacto["baixo"] = fuzz.trimf(x_imp, [0, 0, 40])
    impacto["medio"] = fuzz.trimf(x_imp, [20, 50, 80])
    impacto["alto"] = fuzz.trimf(x_imp, [60, 100, 100])

    # Explorabilidade: Baixa (0-4), Moderada (2-8), Alta (6-10)
    explor["baixa"] = fuzz.trimf(x_exp, [0, 0, 4])
    explor["moderada"] = fuzz.trimf(x_exp, [2, 5, 8])
    explor["alta"] = fuzz.trimf(x_exp, [6, 10, 10])

    # Urgência (saída)
    urg["baixa"] = fuzz.trimf(x_urg, [0, 0, 13])
    urg["media"] = fuzz.trimf(x_urg, [5, 30, 55])
    urg["alta"] = fuzz.trimf(x_urg, [45, 67.5, 90])
    if crit_mf == "tri":
        urg["critica"] = fuzz.trimf(x_urg, [75, 87.5, 100])
    elif crit_mf == "tri100":  # pico em 100 (ombro direito no extremo)
        urg["critica"] = fuzz.trimf(x_urg, [75, 100, 100])
    else:  # Abordagem A: trapézio, ombro de 90 a 100
        urg["critica"] = fuzz.trapmf(x_urg, [75, 90, 100, 100])

    # Matriz de regras 3x3
    regras = [
        # Impacto baixo
        ctrl.Rule(impacto["baixo"] & explor["baixa"], urg["baixa"]),
        ctrl.Rule(impacto["baixo"] & explor["moderada"], urg["baixa"]),
        ctrl.Rule(impacto["baixo"] & explor["alta"], urg["media"]),
        # Impacto médio
        ctrl.Rule(impacto["medio"] & explor["baixa"], urg["media"]),
        ctrl.Rule(impacto["medio"] & explor["moderada"], urg["alta"]),
        ctrl.Rule(impacto["medio"] & explor["alta"], urg["alta"]),
        # Impacto alto
        ctrl.Rule(impacto["alto"] & explor["baixa"], urg["critica"]),
        ctrl.Rule(impacto["alto"] & explor["moderada"], urg["critica"]),
        ctrl.Rule(impacto["alto"] & explor["alta"], urg["critica"]),
    ]
    return ctrl.ControlSystem(regras), (impacto, explor, urg)


def simular(sistema, imp, exp_):
    sim = ctrl.ControlSystemSimulation(sistema)
    sim.input["impacto"] = imp
    sim.input["explorabilidade"] = exp_
    sim.compute()
    return sim.output["urgencia"]


def superficie(sistema, n=21):
    xs = np.linspace(0, 100, n)
    ys = np.linspace(0, 10, n)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    sim = ctrl.ControlSystemSimulation(sistema)
    for i in range(n):
        for j in range(n):
            sim.input["impacto"] = X[i, j]
            sim.input["explorabilidade"] = Y[i, j]
            sim.compute()
            Z[i, j] = sim.output["urgencia"]
    return X, Y, Z


if __name__ == "__main__":
    cenarios = {
        "Baseline  (tri + centroide)": ("tri", "centroid"),
        "Abordagem A (trapmf + centroide)": ("trap", "centroid"),
        "Abordagem B (tri + MOM)": ("tri", "mom"),
        "Abordagem A+B (trapmf + MOM)": ("trap", "mom"),
        "Variante C (pico em 100 + MOM)": ("tri100", "mom"),
        "Variante D (pico em 100 + centroide)": ("tri100", "centroid"),
    }

    print("=" * 72)
    print("MATRIZ DE REGRAS 3x3 (Impacto x Explorabilidade -> Urgência)")
    print("=" * 72)
    print(f"{'Impacto \\ Explorab.':22s}{'Baixa':>12s}{'Moderada':>12s}{'Alta':>12s}")
    print(f"{'Baixo':22s}{'Baixa':>12s}{'Baixa':>12s}{'Média':>12s}")
    print(f"{'Médio':22s}{'Média':>12s}{'Alta':>12s}{'Alta':>12s}")
    print(f"{'Alto':22s}{'Crítica':>12s}{'Crítica':>12s}{'Crítica':>12s}")
    print()
    print("=" * 72)
    print("DESAFIO DOS EXTREMOS")
    print("=" * 72)
    print(f"{'Cenário':38s}{'(0,0)':>10s}{'(100,10)':>12s}{'(50,5)':>10s}")
    sistemas = {}
    for nome, (mf, dz) in cenarios.items():
        s, _ = construir_sistema(mf, dz)
        sistemas[nome] = s
        print(f"{nome:38s}{simular(s, 0, 0):10.2f}{simular(s, 100, 10):12.2f}"
              f"{simular(s, 50, 5):10.2f}")

    # --- Gráfico das funções de pertinência + agregação do baseline -------
    s0, (imp, exp_, urg) = construir_sistema("tri", "centroid")
    fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
    for var, ax, tit in zip((imp, exp_, urg), axs,
                            ("Impacto (%)", "Explorabilidade (0-10)", "Urgência (%)")):
        for termo in var.terms.values():
            ax.plot(var.universe, termo.mf, label=termo.label)
        ax.set_title(tit)
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "funcoes_pertinencia.png", dpi=130)
    print("Imagem salva:", OUT_DIR / "funcoes_pertinencia.png")
    plt.show() if SHOW else None
    plt.close()

    # --- Gráfico do centroide no extremo (100,10) -------------------------
    ativ = np.fmin(1.0, urg["critica"].mf)
    cen = fuzz.defuzz(x_urg, ativ, "centroid")
    plt.figure(figsize=(7, 3.6))
    plt.plot(x_urg, urg["critica"].mf, "k--", label="Crítica (MF)")
    plt.fill_between(x_urg, ativ, alpha=0.3, label="Área agregada")
    plt.axvline(cen, color="r", label=f"Centroide = {cen:.2f}")
    plt.axvline(100, color="gray", ls=":", label="Extremo = 100")
    plt.title("Entrada (100, 10): o centroide não alcança 100")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "centroide_extremo.png", dpi=130)
    print("Imagem salva:", OUT_DIR / "centroide_extremo.png")
    plt.show() if SHOW else None
    plt.close()

    # --- Superfície 3D (baseline centroide) e comparação com MOM ----------
    X, Y, Z = superficie(sistemas["Baseline  (tri + centroide)"])
    _, _, Zm = superficie(sistemas["Abordagem B (tri + MOM)"])
    _, _, Za = superficie(sistemas["Abordagem A (trapmf + centroide)"])

    fig = plt.figure(figsize=(16, 5.5))
    for k, (zz, tit) in enumerate(((Z, "Centroide (triangular)"),
                                   (Za, "Abordagem A: trapmf + centroide"),
                                   (Zm, "Abordagem B: MOM"))):
        ax = fig.add_subplot(1, 3, k + 1, projection="3d")
        surf = ax.plot_surface(X, Y, zz, cmap="viridis", edgecolor="none")
        ax.set_xlabel("Impacto (%)")
        ax.set_ylabel("Explorabilidade")
        ax.set_zlabel("Urgência (%)")
        ax.set_zlim(0, 100)
        ax.set_title(tit)
        ax.view_init(elev=28, azim=-125)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "superficie_decisao.png", dpi=140)
    print("Imagem salva:", OUT_DIR / "superficie_decisao.png")
    plt.show() if SHOW else None
    plt.close()

    # saída principal (somente o baseline) para comprovar continuidade
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", edgecolor="none")
    fig.colorbar(surf, shrink=0.6, label="Urgência (%)")
    ax.set_xlabel("Impacto (%)")
    ax.set_ylabel("Explorabilidade")
    ax.set_zlabel("Urgência (%)")
    ax.set_zlim(0, 100)
    ax.set_title("Superfície de Decisão - Mamdani (centroide)")
    ax.view_init(elev=28, azim=-125)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "superficie_decisao_centroide.png", dpi=140)
    print("Imagem salva:", OUT_DIR / "superficie_decisao_centroide.png")
    plt.show() if SHOW else None
    plt.close()

    # métricas de suavidade: maior salto entre vizinhos da grade
    def salto(zz):
        return max(np.abs(np.diff(zz, axis=0)).max(), np.abs(np.diff(zz, axis=1)).max())
    print("\nSUPERFÍCIE (centroide) - Urgência % | linhas: Explorabilidade, colunas: Impacto")
    idx = [0, 4, 8, 12, 16, 20]
    print("Expl\\Imp " + "".join(f"{X[0, j]:8.0f}" for j in idx))
    for i in idx:
        print(f"{Y[i, 0]:8.0f}  " + "".join(f"{Z[i, j]:8.1f}" for j in idx))
    print("\nMaior salto entre pontos vizinhos da grade (suavidade):")
    print(f"  Centroide triangular : {salto(Z):6.2f}")
    print(f"  A (trapmf+centroide) : {salto(Za):6.2f}")
    print(f"  B (MOM)              : {salto(Zm):6.2f}")
    print(f"Faixa de saída: centroide [{Z.min():.2f}, {Z.max():.2f}] | "
          f"A [{Za.min():.2f}, {Za.max():.2f}] | B [{Zm.min():.2f}, {Zm.max():.2f}]")


MATRIZ DE REGRAS 3x3 (Impacto x Explorabilidade -> Urgência)
Impacto \ Explorab.          Baixa    Moderada        Alta
Baixo                        Baixa       Baixa       Média
Médio                        Média        Alta        Alta
Alto                       Crítica     Crítica     Crítica

DESAFIO DOS EXTREMOS
Cenário                                    (0,0)    (100,10)    (50,5)
Baseline  (tri + centroide)                 4.33       87.50     67.50
Abordagem A (trapmf + centroide)            4.33       90.71     67.50
Abordagem B (tri + MOM)                     0.00       87.50     67.50
Abordagem A+B (trapmf + MOM)                0.00       95.00     67.50
Variante C (pico em 100 + MOM)              0.00      100.00     67.50
Variante D (pico em 100 + centroide)        4.33       91.67     67.50
Imagem salva: /content/funcoes_pertinencia.png
Imagem salva: /content/centroide_extremo.png
Imagem salva: /content/superficie_decisao.png
Imagem salva: /content/superficie_decisao_cent